<a href="https://colab.research.google.com/github/soumyasinghh/TrafficRiskDetection/blob/main/Traffic_Risk_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#installing dependencies
!pip install streamlit ultralytics supervision opencv-python torch torchvision numpy kagglehub pyngrok

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 52.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 67.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 3.1 MB/s eta 0:00:00


In [2]:
code = """
import cv2
import torch
import time
import numpy as np
import supervision as sv
from ultralytics import YOLO
import streamlit as st
import kagglehub
import glob
import random

st.set_page_config(layout="wide")
st.title("Collision Risk Detection")

# input
option = st.radio("Select Input Source", ["Upload Video", "Use Kaggle Dataset"])
video_path = None

# Upload option
if option == "Upload Video":
    uploaded_file = st.file_uploader("Upload a video", type=["mp4","avi","mov"])
    if uploaded_file:
        with open("temp.mp4", "wb") as f:
            f.write(uploaded_file.read())
        video_path = "temp.mp4"

# Dataset option
if option == "Use Kaggle Dataset":
    st.write("Downloading dataset")
    path = kagglehub.dataset_download("arunavfc11/indian-traffic-videos")

    video_files = glob.glob(path + "/**/*.*", recursive=True)
    video_files = [v for v in video_files if v.endswith((".mp4",".avi",".mov"))]

    st.write("Found videos:", len(video_files))

    if len(video_files) > 0:
      video_path = random.choice(video_files)
      st.write("Randomly selected video:")
      st.write(video_path)

    else:
        st.error("No valid video found")

# Start button
run = st.button("Start Processing")

# loading models
@st.cache_resource
def load_model():
    model = YOLO("yolo26n.pt")
    tracker = sv.ByteTrack(frame_rate=30)
    return model, tracker

model, tracker = load_model()

# session state
if "prev_positions" not in st.session_state:
    st.session_state.prev_positions = {}

if "prev_time" not in st.session_state:
    st.session_state.prev_time = time.time()

# helpers
def get_center(box):
    return (int((box[0]+box[2])/2), int((box[1]+box[3])/2))

def euclidean(p1, p2):
    return np.sqrt((p1[0]-p2[0])**2 +
                   (p1[1]-p2[1])**2)

def compute_ttc(p_pos, p_vel, v_pos, v_vel):
    p_pos = np.array(p_pos)
    v_pos = np.array(v_pos)

    rel_pos = v_pos - p_pos
    rel_vel = np.array(v_vel) - np.array(p_vel)

    distance = np.linalg.norm(rel_pos)

    if distance < 1e-5:
        return 0

    closing_speed = np.dot(rel_pos, rel_vel) / distance

    if closing_speed <= 0:
        return 999

    return distance / closing_speed

# processing
if video_path and run:

    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        st.error("Failed to open video")
        st.stop()

    # output video
    output_path = "output.mp4"
    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps == 0:
        fps = 20

    out = cv2.VideoWriter(
        output_path,
        cv2.VideoWriter_fourcc(*'mp4v'),
        fps,
        (640, 360)
    )

    FRAME_WINDOW = st.image([])
    progress = st.progress(0)

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frame_idx = 0

    st.write("Processing started")

    while cap.isOpened():

        ret, frame = cap.read()
        if not ret:
            break

        frame_idx += 1

        # resize
        frame = cv2.resize(frame, (640, 360))

        # skip frames for speed
        if frame_idx % 2 != 0:
            continue

        current_time = time.time()
        dt = current_time - st.session_state.prev_time
        st.session_state.prev_time = current_time

        # detection
        results = model(frame, conf=0.25)[0]
        detections = sv.Detections.from_ultralytics(results)
        detections = tracker.update_with_detections(detections)

        persons, vehicles = [], []

        for i, box in enumerate(detections.xyxy):

            if detections.tracker_id[i] is None:
                continue

            obj_id = detections.tracker_id[i]
            name = model.names[detections.class_id[i]]

            center = get_center(box)
            pos = (center[0], center[1])

            if obj_id in st.session_state.prev_positions:
                prev = st.session_state.prev_positions[obj_id]
                velocity = (np.array(pos) - np.array(prev)) / dt
            else:
                velocity = np.array([0,0])

            st.session_state.prev_positions[obj_id] = pos

            if name == "person":
                persons.append((pos, velocity, center))

            if name in ["car","truck","bus","motorcycle","bicycle"]:
                vehicles.append((pos, velocity, center))

            x1,y1,x2,y2 = map(int, box)
            cv2.rectangle(frame,(x1,y1),(x2,y2),(255,0,0),2)

        # TTC
        for p_pos, p_vel, p_center in persons:
            for v_pos, v_vel, v_center in vehicles:

                dist = euclidean(p_pos, v_pos)

                if dist > 250:
                    continue

                ttc = compute_ttc(p_pos, p_vel, v_pos, v_vel)

                if ttc < 4:
                    risk, color = "HIGH",(0,0,255)
                elif ttc < 8:
                    risk, color = "MEDIUM",(0,165,255)
                else:
                    risk, color = "SAFE",(0,255,0)

                cv2.line(frame, p_center, v_center, color, 2)

                cv2.putText(frame,
                            f"{risk} ({ttc:.1f}s)",
                            (p_center[0], p_center[1]-10),
                            cv2.FONT_HERSHEY_SIMPLEX,
                            0.6, color, 2)

        # save video
        out.write(frame)

        FRAME_WINDOW.image(frame, channels="BGR")
        progress.progress(min(frame_idx / total_frames, 1.0))

        if frame_idx > 500:
            st.write("frame limit reached")
            break

    cap.release()
    out.release()

    st.success("Processing complete")

    with open(output_path, "rb") as f:
        st.download_button("Download Output Video", f, file_name="output.mp4")
"""
with open("app.py", "w") as f:
    f.write(code)

In [23]:
import os, time, requests

# Kill everything
os.system("pkill -f ngrok")
os.system("pkill -f streamlit")
time.sleep(3)

# Start streamlit on port 8502
os.system("nohup streamlit run app.py --server.port 8502 --server.headless true > /tmp/streamlit.log 2>&1 &")
time.sleep(4)

# Tunnel port 8502 — completely fresh, no conflict
os.system("nohup ngrok http 8502 --authtoken 3DGB8623ce9bvg8TaDXEZjezluM_3kJNcpQq5xPWd7LrZAMBA > /tmp/ngrok.log 2>&1 &")
time.sleep(4)

# Get the URL
try:
    tunnels = requests.get("http://localhost:4040/api/tunnels").json()
    public_url = tunnels["tunnels"][0]["public_url"]
    print("Public URL:", public_url)
except Exception as e:
    print("Error:", e)
    os.system("cat /tmp/ngrok.log")

Error: HTTPConnectionPool(host='localhost', port=4040): Max retries exceeded with url: /api/tunnels (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7f457fc19730>: Failed to establish a new connection: [Errno 111] Connection refused'))


In [24]:
import os

print("=== NGROK LOG ===")
os.system("cat /tmp/ngrok.log")

print("\n=== IS NGROK INSTALLED? ===")
os.system("which ngrok && ngrok version")

print("\n=== IS STREAMLIT RUNNING? ===")
os.system("cat /tmp/streamlit.log")

=== NGROK LOG ===

=== IS NGROK INSTALLED? ===

=== IS STREAMLIT RUNNING? ===


0

In [28]:
import requests

NGROK_API_KEY = "3DGGHPLvwf5N4A2NQkJlrmZucld_3aBA8F3sX1FJJFsNruBQc"
headers = {
    "Authorization": f"Bearer {NGROK_API_KEY}",
    "Ngrok-Version": "2"
}

# List all active endpoints
r = requests.get("https://api.ngrok.com/endpoints", headers=headers)
endpoints = r.json()
print(endpoints)

# Delete each one
for ep in endpoints.get("endpoints", []):
    ep_id = ep["id"]
    delete = requests.delete(f"https://api.ngrok.com/endpoints/{ep_id}", headers=headers)
    print(f"Deleted {ep_id}: {delete.status_code}")

{'endpoints': [{'id': 'ep_3DG8w4QrRah3NrjRsnQmMkYsUoj', 'created_at': '2026-05-04T12:21:20Z', 'updated_at': '2026-05-04T12:21:20Z', 'public_url': 'https://cataphyllary-jason-saintly.ngrok-free.dev', 'proto': 'https', 'hostport': 'cataphyllary-jason-saintly.ngrok-free.dev:443', 'type': 'ephemeral', 'domain': {'id': 'rd_3C4muhgnB6L6IMhJXf7VUAKdSqg', 'uri': 'https://api.ngrok.com/reserved_domains/rd_3C4muhgnB6L6IMhJXf7VUAKdSqg'}, 'tunnel': {'id': 'tn_3DG8w4QrRah3NrjRsnQmMkYsUoj', 'uri': 'https://api.ngrok.com/tunnels/tn_3DG8w4QrRah3NrjRsnQmMkYsUoj'}, 'upstream_url': 'http://localhost:8501', 'url': 'https://cataphyllary-jason-saintly.ngrok-free.dev', 'principal': {'id': 'usr_3C4mtk4utAHqfkLj4i20ZoVr7oc', 'uri': ''}, 'bindings': ['public'], 'tunnel_session': {'id': 'ts_3DG57nj0mf11mEWrtoMWzTvf7eG', 'uri': 'https://api.ngrok.com/tunnel_sessions/ts_3DG57nj0mf11mEWrtoMWzTvf7eG'}, 'name': 'http-8501-1c43d9e4-54fa-44d3-83be-bee60eaaa527', 'pooling_enabled': False}], 'uri': 'https://api.ngrok.com

In [30]:
import os, time, requests

time.sleep(5)

os.system("nohup /usr/local/bin/ngrok http 8503 > /tmp/ngrok.log 2>&1 &")
time.sleep(5)

tunnels = requests.get("http://localhost:4040/api/tunnels").json()
print("✅", tunnels["tunnels"][0]["public_url"])

✅ https://cataphyllary-jason-saintly.ngrok-free.dev
